# Public Dataset -> Local Lake Pipeline

This notebook is a step-by-step runbook: `download -> write -> read`.

Target audience: Java/Scala engineering teams. Each section includes practical JVM-oriented notes.


## 0) Install (one-time)

What this step does:
- installs the local package in editable mode.

For Java readers:
- this notebook prepares the local data lake layout; production consumers can read it from JVM tools.


In [ ]:
# Skip if dependencies are already installed
# %pip install -e ..


## 1) Imports

What this step does:
- imports modules for download/write/read operations.

Formats used in this notebook:
- `Parquet`: core columnar file format.
- `Delta`: table format over Parquet with ACID transaction log.
- `Iceberg`: table format with independent metadata layer for multi-engine reads.


In [ ]:
from pathlib import Path
import json
import tempfile
import sys

import pandas as pd

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from kaggle_s3_lake.config import Settings
from kaggle_s3_lake.kaggle_download import download_file_from_url, load_dataframe
from kaggle_s3_lake.writers import write_parquet, write_delta, write_iceberg, dataframe_schema
from kaggle_s3_lake.readers import read_parquet, read_delta, read_iceberg


## 2) Parameters

What this step does:
- defines dataset URL, local paths, namespace, and table names.

What Java teams usually need:
- stable `DATASET_ID` for deterministic path/table naming,
- configurable `LAKE_ROOT` for dev/stage/prod environments.


In [ ]:
SOURCE_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
DATASET_ID = "titanic"
LAKE_ROOT = "./data_lake"
PREFIX = "kaggle-datasets"
NAMESPACE = "kaggle"
CATALOG_NAME = "local"
TABLE_NAME = DATASET_ID.replace("/", "_").replace("-", "_").lower()
SAMPLE_LIMIT = None    # example: 1000


## 3) Runtime setup

What this step does:
- calculates local artifact paths,
- creates directories for Parquet/Delta/Iceberg/manifest.

JVM deployment note:
- keep path structure stable so Spark/Flink jobs do not depend on ad-hoc naming.


In [ ]:
settings = Settings.from_env()
lake_root = Path(LAKE_ROOT).resolve()
dataset_key = DATASET_ID.replace("/", "__")
dataset_root = (lake_root / PREFIX / dataset_key).resolve()

parquet_uri = str((dataset_root / "parquet" / "data.parquet").resolve())
delta_uri = str((dataset_root / "delta").resolve())
warehouse_uri = str((lake_root / PREFIX / "iceberg").resolve())
manifest_path = (dataset_root / "manifest.json").resolve()

Path(parquet_uri).parent.mkdir(parents=True, exist_ok=True)
Path(delta_uri).mkdir(parents=True, exist_ok=True)
Path(warehouse_uri).mkdir(parents=True, exist_ok=True)

print("Parquet:", parquet_uri)
print("Delta:  ", delta_uri)
print("Iceberg warehouse:", warehouse_uri)
print("Manifest:", manifest_path)


## 4) Download dataset from URL

What this step does:
- downloads a public CSV,
- loads it into a DataFrame.

Why this is convenient:
- no Kaggle authentication is required,
- demo can run out of the box.


In [ ]:
tmp_dir = tempfile.TemporaryDirectory(prefix="dataset_download_")
downloaded_file = download_file_from_url(SOURCE_URL, Path(tmp_dir.name))
df, selected_file = load_dataframe([downloaded_file], preferred_file=None)

if SAMPLE_LIMIT:
    df = df.head(SAMPLE_LIMIT).copy()

print(f"selected_file={selected_file}")
print(f"rows={len(df)}, cols={len(df.columns)}")


In [ ]:
display(df.head(10))
display(pd.DataFrame(list(dataframe_schema(df))))


## 5) Write Parquet

What this step does:
- writes a plain Parquet file.

Typical Java/JVM readers:
- Spark SQL (`spark.read.parquet(...)`),
- Flink Table/SQL,
- low-level `parquet-mr`/Hadoop stack.

Maven examples:
- `org.apache.parquet:parquet-hadoop`
- `org.apache.spark:spark-sql_2.12` (match your Scala binary version)


In [ ]:
write_parquet(df, parquet_uri)
print(f"Parquet written: {parquet_uri}")


## 6) Write Delta

What this step does:
- creates a Delta table (`Parquet + _delta_log`).

Typical Java/JVM readers:
- Spark with Delta Lake extension,
- Databricks runtime,
- Delta Kernel for lightweight readers.

Maven example (Spark):
- `io.delta:delta-spark_2.12`

Important:
- ensure Spark and Delta versions are compatible.


In [ ]:
write_delta(df, delta_uri)
print(f"Delta written: {delta_uri}")


## 7) Write Iceberg

What this step does:
- creates an Iceberg table in SQL catalog,
- writes data files + metadata into warehouse.

Typical Java/JVM readers:
- Spark + Iceberg runtime,
- Flink + Iceberg,
- Trino/Presto/Athena through catalog integration.

Maven examples:
- `org.apache.iceberg:iceberg-core`
- `org.apache.iceberg:iceberg-spark-runtime-3.5_2.12` (adapt to your Spark version)


In [ ]:
iceberg_ref = write_iceberg(
    df,
    catalog_name=CATALOG_NAME,
    namespace=NAMESPACE,
    table_name=TABLE_NAME,
    catalog_uri=settings.iceberg_catalog_uri,
    warehouse_uri=warehouse_uri,
)
print(f"Iceberg written: {iceberg_ref}")


## 8) Write manifest.json

What this step does:
- writes a single manifest with paths to all formats and schema snapshot.

For Java teams:
- `manifest.json` can be used as bootstrap metadata for downstream jobs/services.


In [ ]:
manifest = {
    "dataset": DATASET_ID,
    "source_url": SOURCE_URL,
    "source_file": str(selected_file),
    "rows": int(len(df)),
    "schema": list(dataframe_schema(df)),
    "artifacts": {
        "parquet": parquet_uri,
        "delta": delta_uri,
        "iceberg": {
            "catalog": CATALOG_NAME,
            "namespace": NAMESPACE,
            "table": TABLE_NAME,
            "ref": iceberg_ref,
            "catalog_uri": settings.iceberg_catalog_uri,
            "warehouse_uri": warehouse_uri,
        },
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest written: {manifest_path}")


## 9) Read Parquet

What this step does:
- validates that Parquet is readable and returns expected rows/columns.


In [ ]:
df_parquet = read_parquet(parquet_uri)
print(df_parquet.shape)
display(df_parquet.head(10))


## 10) Read Delta

What this step does:
- validates Delta read after write.

For Java Spark pipelines:
- this is where teams usually add smoke-check queries on key columns.


In [ ]:
df_delta = read_delta(delta_uri)
print(df_delta.shape)
display(df_delta.head(10))


## 11) Read Iceberg

What this step does:
- validates catalog-based read using table identifier.

For Java production usage:
- prefer catalog reads (`namespace.table`) over raw file-path access.


In [ ]:
df_iceberg = read_iceberg(
    catalog_name=CATALOG_NAME,
    namespace=NAMESPACE,
    table_name=TABLE_NAME,
    catalog_uri=settings.iceberg_catalog_uri,
    warehouse_uri=warehouse_uri,
)
print(df_iceberg.shape)
display(df_iceberg.head(10))


## 12) Cleanup (optional)

What this step does:
- removes the temporary download directory.

It does not remove `data_lake` artifacts so other services can still read them.


In [ ]:
tmp_dir.cleanup()


## JVM Dependency Cheat Sheet

### Parquet
- Spark: `org.apache.spark:spark-sql_2.12`
- Parquet low-level: `org.apache.parquet:parquet-hadoop`

### Delta
- Spark connector: `io.delta:delta-spark_2.12`
- Keep Spark <-> Delta compatibility in sync

### Iceberg
- Core API: `org.apache.iceberg:iceberg-core`
- Spark runtime: `org.apache.iceberg:iceberg-spark-runtime-<spark_version>_2.12`
- For Flink/Trino, use corresponding Iceberg runtime modules
